# 🔀 Notebook 6: Hybrid Recommender

**Mục tiêu:** Kết hợp SVD + Content-Based

**⚠️ Chạy notebook 00 hoặc 01 trước để có data!**

In [ ]:
import subprocess, sys, os

try:
    import surprise
    import numpy as np
    import pandas as pd
    assert int(np.__version__.split('.')[0]) < 2, 'need numpy<2'
    assert int(pd.__version__.split('.')[0]) < 3, 'need pandas<3'
    print(f'✅ OK (numpy={np.__version__}, pandas={pd.__version__}, surprise={surprise.__version__})')
except Exception as e:
    print(f'📦 Installing... ({e})')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install',
        'numpy<2', 'pandas<3', 'scikit-surprise', 'scikit-learn',
        'matplotlib', 'seaborn', 'tqdm', '-q'])
    print('✅ Install xong! Runtime đang restart...')
    os.kill(os.getpid(), 9)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from surprise import SVD, Dataset, Reader, accuracy
from surprise.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

os.makedirs('results/charts', exist_ok=True)

ratings = pd.read_csv('data/processed/ratings_clean.csv')
movies  = pd.read_csv('data/processed/movies_clean.csv')

reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)

print('✅ Hybrid ready!')

## 2. Train SVD + Content-Based

In [ ]:
# SVD
svd = SVD(n_factors=50, n_epochs=20, random_state=42)
svd.fit(data.build_full_trainset())

# Content-Based
movies['genres_clean'] = movies['genres'].str.replace('|', ' ', regex=False)
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['genres_clean'])
cosine_sim = cosine_similarity(tfidf_matrix)
movie_idx = pd.Series(movies.index, index=movies['movieId'])

print('✅ Both models trained!')

## 3. Hybrid Recommender

In [ ]:
def hybrid_recommend(user_id, ratings_df, movies_df, svd_model, cosine_sim, movie_idx, cf_weight=0.6, top_n=10):
    """Hybrid: α × SVD + (1-α) × Content-Based"""
    user_movies = ratings_df[ratings_df['userId'] == user_id]['movieId'].values
    unseen = [m for m in ratings_df['movieId'].unique() if m not in user_movies]
    top_rated = ratings_df[ratings_df['userId'] == user_id].nlargest(5, 'rating')

    # Pre-compute reference indices
    ref_indices, ref_weights = [], []
    for _, row in top_rated.iterrows():
        if row['movieId'] in movie_idx.index:
            ref_indices.append(movie_idx[row['movieId']])
            ref_weights.append(row['rating'])
    ref_weights = np.array(ref_weights) if ref_weights else np.array([])

    hybrid_scores = {}
    for mid in unseen:
        svd_pred = svd_model.predict(user_id, mid).est
        cb_score = 3.0
        if len(ref_indices) > 0 and mid in movie_idx.index:
            sims = cosine_sim[movie_idx[mid], ref_indices]
            cb_score = np.dot(sims, ref_weights) / len(ref_indices)
        hybrid_scores[mid] = cf_weight * svd_pred + (1 - cf_weight) * cb_score

    sorted_scores = sorted(hybrid_scores.items(), key=lambda x: x[1], reverse=True)
    results = []
    for mid, score in sorted_scores[:top_n]:
        title = movies_df[movies_df['movieId'] == mid]['title'].values
        if len(title) > 0:
            results.append({'movieId': mid, 'title': title[0], 'score': round(score, 2)})
    return results

recs = hybrid_recommend(1, ratings, movies, svd, cosine_sim, movie_idx, cf_weight=0.7)
print('🎬 Top 10 Hybrid (α=0.7):')
pd.DataFrame(recs)

## 4. Thử nghiệm α

In [ ]:
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)
svd2 = SVD(n_factors=50, n_epochs=20, random_state=42)
svd2.fit(trainset)

alpha_values = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
rmse_list = []

for alpha in alpha_values:
    errors = []
    for u, i, r in testset[:5000]:
        svd_pred = svd2.predict(u, i).est
        hybrid_pred = alpha * svd_pred + (1 - alpha) * 3.0
        errors.append((r - hybrid_pred)**2)
    rmse = np.sqrt(np.mean(errors))
    rmse_list.append(rmse)
    print(f'  α={alpha:.1f} → RMSE={rmse:.4f}')

best_idx = np.argmin(rmse_list)
print(f'\n→ Best α = {alpha_values[best_idx]:.1f}')

plt.figure(figsize=(8, 4))
plt.plot(alpha_values, rmse_list, marker='o', color='seagreen', linewidth=2)
plt.scatter([alpha_values[best_idx]], [rmse_list[best_idx]], color='red', s=200, zorder=5, label=f'Best α={alpha_values[best_idx]}')
plt.xlabel('α (CF Weight)')
plt.ylabel('RMSE')
plt.title('Hybrid: RMSE theo α')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('results/charts/06_hybrid_alpha_tuning.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Tổng kết

- Hybrid = tận dụng cả SVD + Content
- α ≈ 0.6–0.8 thường tốt nhất
- Bù đắp nhược điểm cho nhau (SVD → accuracy, Content → cold-start)